# Your robot dog

The dog is already running — you can see it in the panel on the **right**.
This notebook tells it what to do. Run each cell with `Shift`+`Enter`.

Nothing here touches a real robot. The same code will later run on the real
dog by changing one word, but that is switched off on this server.


## 1. Connect to the dog

The dog lives in its own service, so it is already there before you run
anything. `Go2(sim=True)` just connects to it.

`reset()` puts it back at the start — a good habit at the top of a notebook,
because the dog remembers where it was, even if you restart the kernel.


In [ ]:
from go2 import Go2
from go2.notebook import show

dog = Go2(sim=True)
dog.reset()

print(dog.pose())


Watch the panel on the right as you run the next cells — it is the same dog.

> Drag the grey divider to resize. You can also drive the dog by hand with
> the buttons, or `W`/`A`/`S`/`D` after clicking the panel.


## 2. Make it walk

Watch the panel on the right while this runs.

- `vx` forward / backward (m/s)
- `vy` left / right (m/s)
- `wz` turn, counter-clockwise is positive (rad/s)

The command **persists** until you change it, exactly like the real robot,
so `wait()` then `stop()` is the normal shape.


In [ ]:
dog.move(vx=0.4)
dog.wait(4)
dog.stop()

print(dog.pose())


## 3. Where is it?

`pose()` tells you the dog's position and heading in the world.


In [ ]:
p = dog.pose()
print(f'x = {p.x:.2f} m,  y = {p.y:.2f} m,  yaw = {p.yaw:.2f} rad')


## 4. Turn, then walk again

Commands persist until you change them -- exactly like the real robot.


In [ ]:
dog.move(wz=0.8); dog.wait(2); dog.stop()    # turn left
dog.move(vx=0.4); dog.wait(3); dog.stop()    # then forward

print(dog.pose())


## 5. What the dog sees

`camera='head'` is the dog's own forward-facing camera. This is the view
the **real** robot also has -- the outside 'chase' view does not exist on
real hardware.


In [ ]:
show(dog, camera='head')


## 6. Watch it move

The panel on the right is **already live**, so there is nothing to set up:
command the dog and watch it happen.

Asking for forward speed and turn at the same time walks a circle.

> On a laptop install with no panel beside the notebook, `live(dog,
> seconds=6)` draws the same view into the cell output instead. You do not
> need it here -- in this lab it would just render a second copy of what is
> already on your right.

In [ ]:
dog.move(vx=0.3, wz=0.4)   # forward and turning at once -- a circle
dog.wait(6)
dog.stop()

print(dog.pose())

# No panel? (A laptop install, or the panel closed.) Watch it in the cell
# output instead:
#
#     from go2.notebook import live
#     dog.move(vx=0.3, wz=0.4)
#     live(dog, seconds=6)
#     dog.stop()

## 7. Make the dog talk

Click **● record your voice** in the panel on the right, say something,
then click **■ stop**. The dog says it straight back.

Your browser will ask for microphone permission the first time.

Press **▶ play it again** to hear the last clip once more.

You can also play a sound from code:

```python
dog.say('hello.wav')        # a file in your work folder
dog.say(open('hi.webm','rb'))
dog.say_again()             # the last clip, again
```

> The simulated dog has no speaker — MuJoCo does not do sound — so the
> browser watching the dog plays the clip. On the real robot the same
> call goes to the Go2's own speaker.


In [ ]:
# Play back the clip you just recorded, from code this time.
# (Record one in the panel first, or there is nothing to play.)

clip = dog.say_again()
print('played:', clip if clip else 'nothing recorded yet')


In [ ]:
# A sound you can play without a microphone: build a WAV in Python and
# hand it to the dog. `content_type` tells the browser what it is.

import io, math, struct, wave

def beep(seconds=0.6, hz=660, rate=22050):
    """A short tone, as WAV bytes."""
    buf = io.BytesIO()
    with wave.open(buf, 'wb') as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(rate)
        n = int(rate * seconds)
        frames = bytearray()
        for i in range(n):
            fade = min(1.0, min(i, n - i) / (0.05 * rate))   # no click at the ends
            value = 12000 * fade * math.sin(2 * math.pi * hz * i / rate)
            frames += struct.pack('<h', int(value))
        w.writeframes(bytes(frames))
    return buf.getvalue()

dog.say(beep(), content_type='audio/wav')
print('said a beep -- try beep(hz=440) or beep(seconds=1.5)')

### Play a piece of music

`dog_song.mp3` sits in this folder, next to the notebook. It is free music
(CC0 — public domain), so you can do anything you like with it.

Drop your own `.mp3`, `.wav` or `.ogg` into this folder and play that
instead — `dog.say()` takes any of them.

In [ ]:
clip = dog.say('dog_song.mp3')   # 20 seconds, from the folder beside this notebook
print('playing', clip, '-- listen to the panel on the right')

# Your own file works the same way:
#     dog.say('my_tune.mp3')
#
# ...and so does a sound you build yourself, like the beep above.

## 8. Every move a Go2 knows

Your dog can do **four** things here: `stand()`, `sit()`, `move()` and
`stop()`. That is not the whole robot — it is the part the simulator has to
produce by itself, and the part this course needs for navigation and
perception.

The real Go2 has a whole catalogue of moves built into its firmware. They
are what the phone app sends when you tap a button, and they are what
`Go2(sim=False)` will reach when the real-robot backend lands
(see `docs/STEP2-REAL-ROBOT.md` in the project).

Run the next cell to see the list.

> **Flips are not a party trick.** `FrontFlip`, `BackFlip`, `LeftFlip`,
> `HandStand` and the jumps throw 15 kg of robot through the air. They need
> clear space, a flat floor and a person holding the remote, and they can
> hurt the robot or someone standing near it. On a real dog, these happen in
> a booked session with an adult present — never because a cell looked
> interesting.

In [ ]:
# What the simulator does today, and what the real dog's firmware knows.
#
# The numbers are the Go2's own sport-API ids -- the same ones the phone app
# sends. They are here so the list is checkable, not because you need them:
# when the real backend arrives these become ordinary methods.

WORKS_HERE = [
    ('dog.stand()',            'stand still on all four legs'),
    ('dog.sit()',              'lower the body to the ground'),
    ('dog.move(vx, vy, wz)',   'walk at a velocity until told otherwise'),
    ('dog.stop()',             'stop moving, stay standing'),
    ('dog.reset()',            'back to the start: origin, stopped, standing'),
    ('dog.say(sound)',         'play a sound (the browser does it here)'),
    ('dog.backflip()',         'one backwards somersault, then stand'),
]

#  Two marks per row:
#    '='  the simulator has this too (as an animation, like the walking)
#    '!'  on the REAL dog: needs space, a flat floor and an adult
FIRMWARE = [
    ('Move',         1008, 'walk at a velocity',              '='),
    ('StopMove',     1003, 'stop walking',                    '='),
    ('Sit',          1009, 'sit',                             '='),
    ('StandUp',      1004, 'rise from lying down',            '='),
    ('StandDown',    1005, 'lie down',                        '='),
    ('BalanceStand', 1002, 'stand, actively balancing',       '='),
    ('RiseSit',      1010, 'get up from sitting',             ' '),
    ('RecoveryStand',1006, 'get up after a fall',             ' '),
    ('Damp',         1001, 'relax every motor -- the dog DROPS', '!'),
    ('Hello',        1016, 'raise a paw and wave',            ' '),
    ('Stretch',      1017, 'stretch like a dog waking up',    ' '),
    ('WiggleHips',   1033, 'wiggle its hips',                 ' '),
    ('FingerHeart',  1036, 'make a heart shape',              ' '),
    ('Wallow',       1021, 'roll over',                       ' '),
    ('Dance1',       1022, 'dance routine 1',                 ' '),
    ('Dance2',       1023, 'dance routine 2',                 ' '),
    ('Pose',         1028, 'strike a pose',                   ' '),
    ('Scrape',       1029, 'greet / scrape a paw',            ' '),
    ('FrontJump',    1031, 'jump forwards',                   '!'),
    ('FrontPounce',  1032, 'pounce forwards',                 '!'),
    ('FrontFlip',    1030, 'front flip',                      '!'),
    ('BackFlip',     2043, 'back flip',                       '=!'),
    ('LeftFlip',     2041, 'side flip',                       '!'),
    ('HandStand',    2044, 'stand on its front legs',         '!'),
    ('BackStand',    2050, 'stand on its back legs',          '!'),
    ('FreeBound',    2046, 'bound like a rabbit',             '!'),
    ('TrotRun',      1062, 'run',                             '!'),
    ('CrossStep',    2051, 'walk with crossing steps',        ' '),
    ('FreeAvoid',    2048, 'walk while avoiding obstacles',   ' '),
    ('WalkStair',    1049, 'stair-climbing gait',             ' '),
    ('StaticWalk',   1061, 'slow, very stable walk',          ' '),
    ('BodyHeight',   1013, 'stand taller or shorter',         ' '),
    ('SpeedLevel',   1015, 'how fast walking counts as fast', ' '),
]

print('WORKS IN THIS SIMULATOR, RIGHT NOW')
print('-' * 64)
for call, what in WORKS_HERE:
    print(f'  {call:<24} {what}')

print()
print("THE REAL GO2'S FIRMWARE")
print("  '=' the simulator has it too      '!' on the real dog: needs space")
print('-' * 64)
for name, api_id, what, mark in FIRMWARE:
    here = '=' if '=' in mark else ' '
    risk = '!' if '!' in mark else ' '
    print(f'  {here}{risk} {name:<14} {api_id:<5} {what}')

missing = [m for m in FIRMWARE if '=' not in m[3]]
print()
print(f'{len(FIRMWARE)} commands in the firmware; {len(missing)} of them have no')
print('equivalent here yet. Which would you miss most? That is a real')
print('question -- tell your mentor.')

### Try it

The two the simulator has. Watch the panel on the right.

In [ ]:
# The two the simulator does have. Watch the panel on the right.

dog.stand()
dog.wait(1)

dog.sit()          # down
dog.wait(2)

dog.stand()        # and up again
dog.wait(1)

print('sat and stood:', dog.pose())

### Backflip

`dog.backflip()` waits until the dog has landed, so the next line runs after
the flip rather than through the middle of it.

In the simulator this is an **animation**, exactly like the walking: the body
turns through a full circle and the legs tuck in. Nothing is really jumping,
so the dog cannot under-rotate or land badly.

> On the real robot this is the firmware's own trick, and it is the most
> dangerous thing in this whole API — 15 kg of dog leaves the ground and
> comes back down. Real flips happen in a booked session, on a flat floor,
> with clear space and an adult holding the remote.

In [ ]:
dog.stand()
dog.wait(1)

dog.backflip()          # watch the panel on the right!

print('landed:', dog.pose())

## 9. Your turn

Write a loop that walks the dog in a **square**: forward, turn 90 degrees,
four times. Use `dog.pose()` to decide when to stop, not a fixed time.

Hints:
- `math.hypot(x2 - x1, y2 - y1)` gives the distance travelled.
- Angles wrap at +/- pi. `math.atan2(math.sin(d), math.cos(d))` handles that.
- **Always bound your loops.** The dog can be blocked by an obstacle, and a
  loop waiting for a distance that never arrives will hang forever.

A worked solution is in `02_walk_a_square.py`, in this same folder.


In [ ]:
import math

# your code here


## 10. Tidy up

Frees the simulator. Re-run cell 1 to start again.


In [ ]:
from go2.notebook import close_panel

close_panel()
dog.close()
